In [0]:
# ── Gold: Skills Gap ─────────────────────────────────────────────────────────
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, round, avg, sum, count, when, lit, current_timestamp
)

skills_df    = spark.table("silver.skills")
employees_df = spark.table("silver.employees")
projects_df  = spark.table("silver.projects")

joined_df = (
    skills_df
    .join(employees_df.select("employee_id", "practice_area", "seniority"),
          on="employee_id", how="left")
)

skills_summary_df = (
    joined_df
    .groupBy("skill")
    .agg(
        count("employee_id").alias("total_employees"),
        count(when(col("certified") == True, True)).alias("certified_count"),
        count(when(col("certified") == False, True)).alias("uncertified_count"),
        count(when(col("proficiency") == "Expert", True)).alias("expert_count"),
        count(when(col("proficiency") == "Advanced", True)).alias("advanced_count"),
        count(when(col("proficiency") == "Intermediate", True)).alias("intermediate_count"),
        count(when(col("proficiency") == "Beginner", True)).alias("beginner_count"),
        round(avg(when(col("proficiency") == "Expert", lit(4))
            .when(col("proficiency") == "Advanced", lit(3))
            .when(col("proficiency") == "Intermediate", lit(2))
            .when(col("proficiency") == "Beginner", lit(1))), 2).alias("avg_proficiency_score")
    )
    .withColumn("certification_rate_pct", round(
        (col("certified_count") / col("total_employees")) * 100, 2))
    .withColumn("coverage_flag", when(
        col("total_employees") < 10, lit("Underrepresented"))
        .when(col("total_employees") < 20, lit("Moderate"))
        .otherwise(lit("Well covered")))
    .withColumn("gold_loaded_at", current_timestamp())
)

practice_skills_df = (
    joined_df
    .groupBy("practice_area", "skill")
    .agg(
        count("employee_id").alias("total_employees"),
        count(when(col("certified") == True, True)).alias("certified_count"),
        count(when(col("proficiency").isin(["Advanced", "Expert"]), True)).alias("senior_proficiency_count"),
        round(avg(when(col("proficiency") == "Expert", lit(4))
            .when(col("proficiency") == "Advanced", lit(3))
            .when(col("proficiency") == "Intermediate", lit(2))
            .when(col("proficiency") == "Beginner", lit(1))), 2).alias("avg_proficiency_score")
    )
    .withColumn("certification_rate_pct", round(
        (col("certified_count") / col("total_employees")) * 100, 2))
    .withColumn("gold_loaded_at", current_timestamp())
)

# Write to Gold
skills_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.skills_summary")
practice_skills_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.skills_by_practice")

print(f"Written to gold.skills_summary:      {spark.table('gold.skills_summary').count()} rows")
print(f"Written to gold.skills_by_practice:  {spark.table('gold.skills_by_practice').count()} rows")

In [0]:

spark.sql("USE CATALOG workspace")

tables = [
    "utilisation_monthly",
    "utilisation_by_practice",
    "bench_by_employee",
    "bench_by_practice",
    "skills_summary",
    "skills_by_practice"
]

print("=== Gold Layer Summary ===\n")
for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM gold.{table}").collect()[0][0]
    print(f"gold.{table}: {count} rows")

In [0]:
spark.sql("USE CATALOG workspace")
display(spark.sql("SHOW TABLES IN gold"))